In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
from enum import Enum

class Strategy(Enum):
    THREESOLDIERS = 1
    ENGULFING = 2
    
class Audit(Enum):
    WEEKLY = 1
    MONTHLY = 2
    
class Interval(Enum):
    HOURLY_ONEHOUR = 1
    HOURLY_FOURHOUR = 2
    DAILY_ONEDAY = 3
    
engulfing_body_condition = 2
threesoldiers_body_condition = [0.5,0.5]
#march and nov 21 and march 24 almost same price (all high)
# 3         11           3

In [3]:
file = "../../data/csv/Binance_BTCUSDT_1h.csv"
# enums
strategy = Strategy.ENGULFING
audit = Audit.MONTHLY
interval = Interval.HOURLY_ONEHOUR

usdt_balance = 1000
buy_percentage_init = 10
sell_percentage_init = 13
learning_rate = 1.3
bitcoin_balance: np.longdouble = 0
period = 7 # as candles
drop_rows = True
row_upper_limit = 365 * 2 - 1 # datas upper than
row_lower_limit = 0 # datas lower than
add_each_month = 0 #20 # dollars
period_counter = 1 # add money after n periods

total_balance = usdt_balance + bitcoin_balance
buyPositionsNumber = 0
sellPositionsNumber = 0
buyDuplicateChecker = 0
sellDuplicateChecker = 0
buy_percentage_current = buy_percentage_init
sell_percentage_current = sell_percentage_init
usdt_balance_init = usdt_balance

print(f'chosen strategy: {strategy.name}')

chosen strategy: ENGULFING


In [4]:
# if columns are not set properly, open the file one time and save it under .csv format
# and try again
if drop_rows:
    if row_lower_limit != 0:
        df = pd.read_csv(file,skiprows=row_lower_limit,nrows=(row_upper_limit - row_lower_limit + 1),names = ['Unix' ,'Date' ,'Symbol' ,'Open' ,'High' ,'Low' ,'Close' ,'Volume BTC','Volume USDT','tradecount'])
    else:
        df = pd.read_csv(file,skiprows=row_lower_limit,nrows=(row_upper_limit - row_lower_limit + 1))
       
else:
    df = pd.read_csv(file)

df.head()


,https://www.CryptoDataDownload.com,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9
0,Unix,Date,Symbol,Open,High,Low,Close,Volume BTC,Volume USDT,tradecount
1,1.7102E+12,3/11/2024 23:00,BTCUSDT,72441.1,72444.66,72053.94,72078.1,1380.1613,99686868.25,53325
2,1.71019E+12,3/11/2024 22:00,BTCUSDT,72517.32,72750,72392.57,72441.11,1838.21988,133430058.7,66440
3,1.71019E+12,3/11/2024 21:00,BTCUSDT,72115.77,72594.13,72038.31,72517.32,1118.88133,80867124.95,55976
4,1.71019E+12,3/11/2024 20:00,BTCUSDT,71970,72261.1,71836.36,72115.76,1276.43156,92022576.13,56026


In [5]:
df.describe()

,https://www.CryptoDataDownload.com,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9
count,730,730,730,730,730,730,730,730,730,730
unique,265,730,2,728,715,718,729,730,730,729
top,1.70889E+12,Date,BTCUSDT,68300,52494,51500,51090.01,Volume BTC,Volume USDT,64638
freq,3,1,729,2,2,3,2,1,1,2


In [6]:
df.columns

Index(['https://www.CryptoDataDownload.com', 'Unnamed: 1', 'Unnamed: 2',
       'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7',
       'Unnamed: 8', 'Unnamed: 9'],
      dtype='object')

In [7]:
df.iloc[0]

https://www.CryptoDataDownload.com           Unix
Unnamed: 1                                   Date
Unnamed: 2                                 Symbol
Unnamed: 3                                   Open
Unnamed: 4                                   High
Unnamed: 5                                    Low
Unnamed: 6                                  Close
Unnamed: 7                             Volume BTC
Unnamed: 8                            Volume USDT
Unnamed: 9                             tradecount
Name: 0, dtype: object

In [8]:
if row_lower_limit == 0 and drop_rows:
    df.columns = df.iloc[0]
    df = df.drop(0)
    df.head()

In [9]:
cdf = df[['Date','Open','High','Low','Close']]
cdf.head()

,Date,Open,High,Low,Close
1,3/11/2024 23:00,72441.1,72444.66,72053.94,72078.1
2,3/11/2024 22:00,72517.32,72750,72392.57,72441.11
3,3/11/2024 21:00,72115.77,72594.13,72038.31,72517.32
4,3/11/2024 20:00,71970,72261.1,71836.36,72115.76
5,3/11/2024 19:00,72479.99,72800,71760,71970


In [10]:
cdf.describe()

,Date,Open,High,Low,Close
count,729,729,729,729,729
unique,729,727,714,717,728
top,3/11/2024 23:00,51090.01,57200,51500,51090.01
freq,1,2,2,3,2


In [11]:
print(type(cdf['Open'][1]))
print(type(cdf['High'][1]))

<class 'str'>
<class 'str'>


In [12]:
import warnings
warnings.filterwarnings('ignore')

cdf['Open'] = pd.to_numeric(cdf['Open'])
cdf['High'] = pd.to_numeric(cdf['High'])
cdf['Low'] = pd.to_numeric(cdf['Low'])
cdf['Close'] = pd.to_numeric(cdf['Close'])

print(type(cdf['Open'][1]))
print(type(cdf['High'][1]))

<class 'numpy.float64'>
<class 'numpy.float64'>


In [13]:
cdf['isGreen'] = cdf["Close"]
cdf['bodyLen'] = abs(cdf["Close"] - cdf["Open"])
cdf['bodyLen_normalised'] = abs(cdf["Close"] - cdf["Open"]) / cdf['Open']
cdf = cdf.reset_index()
del cdf['index']
cdf.head()

,Date,Open,High,Low,Close,isGreen,bodyLen,bodyLen_normalised
0,3/11/2024 23:00,72441.10,72444.66,72053.94,72078.10,72078.10,363.00,0.005011
1,3/11/2024 22:00,72517.32,72750.00,72392.57,72441.11,72441.11,76.21,0.001051
2,3/11/2024 21:00,72115.77,72594.13,72038.31,72517.32,72517.32,401.55,0.005568
3,3/11/2024 20:00,71970.00,72261.10,71836.36,72115.76,72115.76,145.76,0.002025
4,3/11/2024 19:00,72479.99,72800.00,71760.00,71970.00,71970.00,509.99,0.007036


In [14]:
cdf.describe()

,Open,High,Low,Close,isGreen,bodyLen,bodyLen_normalised
count,729.000000,729.000000,729.000000,729.000000,729.000000,729.000000,729.000000
mean,57204.753731,57454.451879,56962.461550,57239.974691,57239.974691,231.609602,0.003957
std,7352.226513,7440.639882,7266.989663,7365.261619,7365.261619,279.009641,0.004530
min,47196.490000,47337.920000,47105.440000,47196.480000,47196.480000,0.000000,0.000000
25%,51509.890000,51638.900000,51311.000000,51509.990000,51509.990000,64.200000,0.001164
50%,52213.870000,52380.000000,52085.240000,52217.730000,52217.730000,147.000000,0.002604
75%,63357.990000,63780.060000,63088.250000,63397.990000,63397.990000,288.190000,0.004906
max,72517.320000,72800.000000,72392.570000,72517.320000,72517.320000,2577.390000,0.040279


In [15]:
del cdf['High']
del cdf['Low']

for i in range(len(cdf)):
    tmp = cdf.loc[i,cdf.columns.values]
    if tmp['Open'] <= tmp['Close']:
        tmp['isGreen'] = 1
    else:
        tmp['isGreen'] = 0

    cdf.loc[i] = tmp.values
        
cdf.head(10)

,Date,Open,Close,isGreen,bodyLen,bodyLen_normalised
0,3/11/2024 23:00,72441.10,72078.10,0.0,363.00,0.005011
1,3/11/2024 22:00,72517.32,72441.11,0.0,76.21,0.001051
2,3/11/2024 21:00,72115.77,72517.32,1.0,401.55,0.005568
3,3/11/2024 20:00,71970.00,72115.76,1.0,145.76,0.002025
4,3/11/2024 19:00,72479.99,71970.00,0.0,509.99,0.007036
5,3/11/2024 18:00,72481.11,72480.00,0.0,1.11,0.000015
6,3/11/2024 17:00,72501.11,72481.11,0.0,20.00,0.000276
7,3/11/2024 16:00,72010.01,72501.10,1.0,491.09,0.006820
8,3/11/2024 15:00,72170.69,72010.00,0.0,160.69,0.002227
9,3/11/2024 14:00,71503.97,72170.68,1.0,666.71,0.009324


In [16]:
cdf.tail()

,Date,Open,Close,isGreen,bodyLen,bodyLen_normalised
724,2/10/2024 17:00,47249.99,47196.48,0.0,53.51,0.001132
725,2/10/2024 16:00,47353.48,47250.00,0.0,103.48,0.002185
726,2/10/2024 15:00,47270.00,47353.48,1.0,83.48,0.001766
727,2/10/2024 14:00,47303.58,47270.00,0.0,33.58,0.000710
728,2/10/2024 13:00,47231.54,47303.59,1.0,72.05,0.001525


In [17]:
def updateTotalBalance(i):
    global usdt_balance, bitcoin_balance, total_balance
    total_balance = usdt_balance + bitcoin_balance * cdf.loc[i]["Close"]

In [18]:
def buyBTC(i):
    global usdt_balance, bitcoin_balance, total_balance,buyPositionsNumber, sellPositionsNumber
    global buyDuplicateChecker,sellDuplicateChecker, learning_rate, buy_percentage_current
    global buy_percentage_init, sell_percentage_current, sell_percentage_init
   
    buyDuplicateChecker += 1
    sellDuplicateChecker = 0
    if buyDuplicateChecker != 3:
        return
    
    buyDuplicateChecker = 0
    buyDuplicateChecker = 0
    print("\033[92m \033[1m{}\033[00m" .format(f"buy opportunity!\n"))        

    if usdt_balance > 0:
        buyPositionsNumber += 1
        
        print(f'current buy percentage: {buy_percentage_current}')
        amountToBuy_usdt = total_balance * buy_percentage_current / 100
        buy_percentage_current *= learning_rate
        sell_percentage_current = sell_percentage_init
        
        if amountToBuy_usdt > usdt_balance:
            amountToBuy_usdt = usdt_balance
            
        addedBitcoinAmount = np.longdouble(amountToBuy_usdt / cdf.loc[i]["Close"])
        usdt_balance -= amountToBuy_usdt
        bitcoin_balance += addedBitcoinAmount
        updateTotalBalance(i)
        print(f'\tbought {amountToBuy_usdt} $ BTC ({addedBitcoinAmount} BTCs) in {cdf.loc[i]["Date"]}')
        print(f"\tnew usdt balance: {usdt_balance}")
        print(f'\tnew bitcoin balance: {bitcoin_balance} ({bitcoin_balance * cdf.loc[i]["Close"]} $)')
        
    else:
        print(f"\tthere isn't any usdt to buy BTC with it!")
        print(f'\tcurrent BTC balance: {bitcoin_balance} ({bitcoin_balance * cdf.loc[i]["Close"]} $)')

In [19]:
def sellBTC(i):
    global usdt_balance, bitcoin_balance, total_balance,buyPositionsNumber, sellPositionsNumber
    global sellDuplicateChecker,buyDuplicateChecker, buy_percentage_current, buy_percentage_init
    global sell_percentage_current, sell_percentage_init, learning_rate
    
    sellDuplicateChecker += 1
    buyDuplicateChecker = 0
    if sellDuplicateChecker != 3:
        return
    
    buyDuplicateChecker = 0
    sellDuplicateChecker = 0
    print("\033[91m \033[1m{}\033[00m" .format(f"sell opportunity!\n"))        
    if bitcoin_balance > 0:
        sellPositionsNumber += 1
        
        currentValueOfAllBitcoins = bitcoin_balance * cdf.loc[i]["Close"]
        
        print(f'current sell percentage: {sell_percentage_current}')
        amountToSell_usdt = total_balance * sell_percentage_current / 100
        sell_percentage_current *= learning_rate
        buy_percentage_current = buy_percentage_init
        
        if amountToSell_usdt > currentValueOfAllBitcoins:
            amountToSell_usdt = currentValueOfAllBitcoins
        
        amountToSell_bitcoin = amountToSell_usdt / cdf.loc[i]["Close"]
        usdt_balance += amountToSell_usdt
        bitcoin_balance -= amountToSell_bitcoin
        updateTotalBalance(i)
        print(f'\tsold {amountToSell_usdt} $ of BTC ({amountToSell_usdt} $) in {cdf.loc[i]["Date"]}')
        print(f"\tnew usdt balance: {usdt_balance}")
        print(f'\tnew bitcoin balance: {bitcoin_balance} ({bitcoin_balance * cdf.loc[i]["Close"]} $)')
        
    else:
        print(f"\tthere isn't any bitcoin to sell!")
        print(f'\tcurrent USDT balance: {usdt_balance} $')

In [20]:
def checkForPosition(i,strategy):
    if strategy == Strategy.THREESOLDIERS:
        if i >= len(cdf) - 2:
            return
        suitableForBuy = False
        suitableForSell = False
    #     print(f'i = {i}:')

        if cdf.loc[0 + i]['isGreen'] == 1.0:
            if cdf.loc[1 + i]['isGreen'] == 1.0:
                if cdf.loc[2 + i]['isGreen'] == 1.0:
                    if cdf.loc[1 + i]['bodyLen'] / cdf.loc[0 + i]['bodyLen'] >= threesoldiers_body_condition[0]:
                        if cdf.loc[2 + i]['bodyLen'] / cdf.loc[1 + i]['bodyLen'] >= threesoldiers_body_condition[1]:
                            suitableForBuy = True
                
        if cdf.loc[0 + i]['isGreen'] == 0.0:
            if cdf.loc[1 + i]['isGreen'] == 0.0:
                if cdf.loc[2 + i]['isGreen'] == 0.0:
                    if cdf.loc[1 + i]['bodyLen'] / cdf.loc[0 + i]['bodyLen'] >= threesoldiers_body_condition[0]:
                        if cdf.loc[2 + i]['bodyLen'] / cdf.loc[1 + i]['bodyLen'] >= threesoldiers_body_condition[1]:
                            suitableForSell = True
                
    if strategy == Strategy.ENGULFING:
        if i >= len(cdf) - 1:
            return
        suitableForBuy = False
        suitableForSell = False
    #     print(f'i = {i}:')

        if cdf.loc[0 + i]['isGreen'] == 1.0:
            if cdf.loc[1 + i]['isGreen'] == 0.0:
                    if cdf.loc[1 + i]['bodyLen'] / cdf.loc[0 + i]['bodyLen'] >= engulfing_body_condition:
                        suitableForBuy = True
                
        if cdf.loc[0 + i]['isGreen'] == 1.0:
            if cdf.loc[1 + i]['isGreen'] == 0.0:
                if cdf.loc[2 + i]['isGreen'] == 0.0:
                    if cdf.loc[1 + i]['bodyLen'] / cdf.loc[0 + i]['bodyLen'] >= engulfing_body_condition:
                        suitableForSell = True
                

        if suitableForBuy:
    #         print(f'{i} is suitable for buy!')
            buyBTC(i)

        if suitableForSell:
    #         print(f"{i} is suitable for sell!")
            sellBTC(i)

    #     if not (suitableForBuy or suitableForSell):
    #         print(f'{i} is not suitable for buy nor sell!')

In [21]:
def timeout(i):
    global usdt_balance, bitcoin_balance, total_balance,buyPositionsNumber, sellPositionsNumber, counter
    updateTotalBalance(i)
    print("\033[91m \033[1m{}\033[00m" .format(f"\n\t\tCOUNTER REACHER IT'S MAXIMUM!\n"))
    print("\033[91m \033[1m{}\033[00m" .format(f"\t\topened {buyPositionsNumber} buy\
positions and {sellPositionsNumber} sell positions!\n"))
    print("\033[91m \033[1m{}\033[00m" .format(f"\t\tcurrent usdt balance: {usdt_balance}\n"))
    print("\033[91m \033[1m{}\033[00m" .format(f"\t\tcurrent bitcoin balance: {bitcoin_balance} ({bitcoin_balance * cdf.loc[i]['Close']} $)\n"))
    print("\033[91m \033[1m{}\033[00m" .format(f"\t\tcurrent total balance: {total_balance}\n"))
    counter = 0
    buyPositionsNumber = 0
    sellPositionsNumber = 0
    return total_balance

In [22]:
def calculatePeriod():
    global audit, interval, period
    if audit == Audit.MONTHLY:
        if interval == Interval.DAILY_ONEDAY:
            period = 30
        if interval == Interval.HOURLY_FOURHOUR:
            period = 6 * 30
        if interval == Interval.HOURLY_ONEHOUR:
            period = 24 * 30
            
    if audit == Audit.WEEKLY:
        if interval == Interval.DAILY_ONEDAY:
            period = 7
        if interval == Interval.HOURLY_FOURHOUR:
            period = 6 * 7
        if interval == Interval.HOURLY_ONEHOUR:
            period = 24 * 7

In [23]:
totals = []
counter = 0
total_counter = 0

calculatePeriod()
for i in range(len(cdf) -1, -1, -1):
    checkForPosition(i,strategy)
    counter += 1
    if counter == period:
        totals.append(timeout(i))
        total_counter += 1
    if total_counter == period_counter:
        print(f"new month!,\n\t{add_each_month} $ added to USDT balance.")
        usdt_balance += add_each_month
        total_counter = 0

 buy opportunity!

current buy percentage: 10
	bought 100.0 $ BTC (0.0020801619530890197 BTCs) in 2/12/2024 5:00
	new usdt balance: 900.0
	new bitcoin balance: 0.0020801619530890197 (100.0 $)
 buy opportunity!

current buy percentage: 13.0
	bought 130.0 $ BTC (0.002507812801419808 BTCs) in 2/16/2024 20:00
	new usdt balance: 770.0
	new bitcoin balance: 0.004587974754508828 (237.8314353242286 $)
 buy opportunity!

current buy percentage: 16.900000000000002
	bought 170.32351256979464 $ BTC (0.003289622847840595 BTCs) in 2/18/2024 20:00
	new usdt balance: 599.6764874302054
	new bitcoin balance: 0.007877597602349422 (407.8704934592437 $)
 buy opportunity!

current buy percentage: 21.970000000000002
	bought 221.35807170141197 $ BTC (0.004254922184019144 BTCs) in 2/20/2024 21:00
	new usdt balance: 378.3184157287934
	new bitcoin balance: 0.012132519786368565 (631.1822093660383 $)
 buy opportunity!

current buy percentage: 28.561000000000003
	bought 288.3234735333349 $ BTC (0.005584959406933249

In [24]:
print("totals:")
for i in totals:
    print(i)
print(f"min: {min(totals)}, max: {max(totals)}")
# print(f"all buy and no change: {usdt_balance_init * (cdf.iloc[0]['Close'] / cdf.iloc[len(cdf) - 1]['Close'])}")

totals:
1395.209797811703
min: 1395.209797811703, max: 1395.209797811703


### ############################################################

In [25]:
min(totals)

1395.209797811703